# YOLOv8m Random Search: Cross Year

This notebook runs the cross year Random Search experiment for YOLOv8m using the Year2021 and Year2022 weed datasets.

Protocol:

1. Change only `TRAIN_YEAR` and `TEST_YEAR` to reverse the cross year direction.
2. Hyperparameter optimization uses the source year training and validation data only.
3. The target year test split is never used to rank or select hyperparameters.
4. Random Search evaluates 8 independent configurations for 40 epochs each, giving exactly 320 HPO epoch units.
5. Selection uses source validation mAP50-95 only.
6. The search dimensions are `lr0`, `lrf`, `momentum`, `weight_decay`, and `scale`.
7. The selected configuration is frozen and retrained with seeds 42, 123, and 999.
8. Final evaluation is performed on the corresponding target year test split.
9. When Year2021 is the source year, only the source training split is stratified subsampled to match the target year training split size.
10. Runtime settings use `workers=8`, `cache="disk"`, `batch=16`, `imgsz=640`, and `patience=0`.
11. The 120 final training epoch units are reported separately from the 320 HPO epoch units.


## Module 1: Imports, paths, experiment direction, and fixed protocol

This module defines the portable project layout, cross year direction, study classes, split seeds, exact 320 epoch Random Search budget, 5D search space, shared YOLO settings, and privacy safe path helpers used by later modules.


In [ ]:
from pathlib import Path
from datetime import datetime
from statistics import mean, stdev
from typing import Dict, Any, List, Optional, Tuple
from collections import Counter, defaultdict

import csv
import json
import math
import os
import random
import shutil
import sys
import time
import traceback
import uuid
import xml.etree.ElementTree as ET

import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO


# Cross year direction
TRAIN_YEAR = "2021"
TEST_YEAR = "2022"

VALID_YEARS = {"2021", "2022"}
if TRAIN_YEAR not in VALID_YEARS or TEST_YEAR not in VALID_YEARS:
    raise ValueError("TRAIN_YEAR and TEST_YEAR must each be '2021' or '2022'.")
if TRAIN_YEAR == TEST_YEAR:
    raise ValueError("Cross year evaluation requires different train and test years.")

TRAIN_DATASET_TAG = f"Year{TRAIN_YEAR}"
TEST_DATASET_TAG = f"Year{TEST_YEAR}"
CROSS_YEAR_TAG = f"Y{TRAIN_YEAR}_to_Y{TEST_YEAR}"


# Portable project layout
# Expected dataset locations:
# ./Datasets/Year2021
# ./Datasets/Year2022
WORKSPACE_ROOT = Path.cwd()
DATASETS_ROOT = WORKSPACE_ROOT / "Datasets"
TRAIN_SOURCE_ROOT = DATASETS_ROOT / TRAIN_DATASET_TAG
TEST_TARGET_ROOT = DATASETS_ROOT / TEST_DATASET_TAG

PROJECT_ROOT = (
    WORKSPACE_ROOT
    / "outputs"
    / f"yolov8m_random_search_cross_year_{CROSS_YEAR_TAG.lower()}"
)
RUNS_ROOT = PROJECT_ROOT / "runs"
SPLITS_ROOT = PROJECT_ROOT / "splits"
STATS_ROOT = PROJECT_ROOT / "stats"
YOLO_DATA_ROOT = PROJECT_ROOT / "yolo_data"


# Target classes
CLASS_NAMES = [
    "Waterhemp",
    "Carpetweed",
    "Morningglory",
    "Goosegrass",
    "SpottedSpurge",
    "PalmerAmaranth",
    "Purslane",
    "Ragweed",
]
TARGET_CLASS_SET = set(CLASS_NAMES)

LABEL_ALIASES = {
    "morninglory": "Morningglory",
    "morning glory": "Morningglory",
    "morningglory": "Morningglory",
    "spotted spurge": "SpottedSpurge",
    "spottedspurge": "SpottedSpurge",
    "palmer amaranth": "PalmerAmaranth",
    "palmeramaranth": "PalmerAmaranth",
}


# Split protocol
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15

OPTIMIZATION_SPLIT_SEED = 42
OPTIMIZATION_TRAIN_SEED = 42
FINAL_EVALUATION_SEEDS = [42, 123, 999]


# Year2021 source training subsampling protocol
SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE = True
SUBSAMPLE_TARGET_MODE = "target_train_size"
SUBSAMPLE_SEED_OFFSET = 10000


# Shared YOLO runtime settings
MODEL_NAME = "yolov8m.pt"
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 8
PATIENCE = 0
IMAGE_PLACEMENT_MODE = "symlink"
CACHE_MODE = "disk"

HPO_EPOCHS = 40
HPO_WARMUP_EPOCHS = 3.0
HPO_CLOSE_MOSAIC = 10

FINAL_EPOCHS = 40
FINAL_WARMUP_EPOCHS = 3.0
FINAL_CLOSE_MOSAIC = 10

N_RANDOM_TRIALS = 8
RANDOM_SEARCH_SEED = 123

HPO_EPOCH_UNITS = N_RANDOM_TRIALS * HPO_EPOCHS
FINAL_EVALUATION_EPOCH_UNITS = (
    len(FINAL_EVALUATION_SEEDS) * FINAL_EPOCHS
)

if HPO_EPOCH_UNITS != 320:
    raise RuntimeError(
        "Random Search HPO budget must be exactly 8 x 40 = 320 epoch units."
    )


# Focused 5D search space
SEARCH_SPACE_VERSION = "Focused5D_CrossYear_RS8x40_mAP5095_v1"
SEARCH_SPACE = {
    "learning_rate": {
        "low": 0.0010,
        "high": 0.0100,
        "distribution": "log",
    },
    "lrf": {
        "low": 0.0100,
        "high": 0.2000,
        "distribution": "log",
    },
    "momentum": {
        "low": 0.8500,
        "high": 0.9700,
        "distribution": "linear",
    },
    "weight_decay": {
        "low": 0.0001,
        "high": 0.0010,
        "distribution": "log",
    },
    "scale": {
        "low": 0.2000,
        "high": 0.8000,
        "distribution": "linear",
    },
}
SEARCH_PARAMETER_ORDER = [
    "learning_rate",
    "lrf",
    "momentum",
    "weight_decay",
    "scale",
]

REFERENCE_UNTUNED_VALUES = {
    "learning_rate": 0.010,
    "lrf": 0.010,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "scale": 0.50,
}

for parameter_name, reference_value in REFERENCE_UNTUNED_VALUES.items():
    parameter_spec = SEARCH_SPACE[parameter_name]
    if not (
        float(parameter_spec["low"])
        <= float(reference_value)
        <= float(parameter_spec["high"])
    ):
        raise RuntimeError(
            f"Reference value {parameter_name}={reference_value} is outside "
            f"[{parameter_spec['low']}, {parameter_spec['high']}]."
        )


# Fixed settings that are not tuned
FIXED_TRAINING_ARGS = {
    "optimizer": "SGD",
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "nbs": 64,
    "mosaic": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.70,
    "hsv_v": 0.40,
    "degrees": 0.0,
    "translate": 0.10,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mixup": 0.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "cache": CACHE_MODE,
    "cos_lr": False,
    "rect": False,
}

STRICT_FITNESS_ALIGNMENT_CHECK = True
FITNESS_ALIGNMENT_TOLERANCE = 0.002
REQUIRE_ALL_HPO_TRIALS = True


# Run output folders
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = RUNS_ROOT / "rs8x40" / RUN_TIMESTAMP
HPO_TRIALS_ROOT = RUN_ROOT / "hpo"
FINAL_TRAINING_ROOT = RUN_ROOT / "final40"
TEST_EVAL_ROOT = RUN_ROOT / "target_test"
RUN_STATS_ROOT = RUN_ROOT / "stats"
RUN_PLOTS_ROOT = RUN_ROOT / "plots"

for folder in [
    PROJECT_ROOT,
    RUNS_ROOT,
    SPLITS_ROOT,
    STATS_ROOT,
    YOLO_DATA_ROOT,
    RUN_ROOT,
    HPO_TRIALS_ROOT,
    FINAL_TRAINING_ROOT,
    TEST_EVAL_ROOT,
    RUN_STATS_ROOT,
    RUN_PLOTS_ROOT,
]:
    folder.mkdir(parents=True, exist_ok=True)


def privacy_safe_text(
    value: Any,
    workspace_root: Path,
) -> str:
    """Remove the absolute workspace prefix from persisted error text."""
    text = str(value)
    workspace_text = str(workspace_root.resolve())
    return text.replace(workspace_text, "<WORKSPACE>")


def relative_output_path(
    path: Path,
    workspace_root: Path,
) -> str:
    """Return a portable path for saved manifests and console output."""
    path = Path(path)
    try:
        return path.resolve().relative_to(
            workspace_root.resolve()
        ).as_posix()
    except ValueError:
        return path.name


print("=" * 88)
print("YOLOv8m RANDOM SEARCH: CROSS YEAR")
print("=" * 88)
print("Direction:", CROSS_YEAR_TAG)
print("Source dataset:", f"Datasets/{TRAIN_DATASET_TAG}")
print("Target dataset:", f"Datasets/{TEST_DATASET_TAG}")
print("Model:", MODEL_NAME)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Workers:", WORKERS)
print("Cache:", CACHE_MODE)
print("HPO fitness: source validation mAP50-95 only")
print(
    "HPO budget:",
    N_RANDOM_TRIALS,
    "x",
    HPO_EPOCHS,
    "=",
    HPO_EPOCH_UNITS,
    "epoch units",
)
print(
    "Final evaluation budget:",
    FINAL_EVALUATION_EPOCH_UNITS,
    "epoch units, reported separately",
)
print("Optimization split seed:", OPTIMIZATION_SPLIT_SEED)
print("Optimization training seed:", OPTIMIZATION_TRAIN_SEED)
print("Random Search seed:", RANDOM_SEARCH_SEED)
print("Final seeds:", FINAL_EVALUATION_SEEDS)
print("Python:", sys.version.split()[0])
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print(
    "Output root:",
    relative_output_path(RUN_ROOT, WORKSPACE_ROOT),
)
print("=" * 88)


## Module 2: Annotation parsing and normalized record helpers

This module reads Pascal VOC XML and common JSON annotation layouts, normalizes class aliases, reads image dimensions when needed, and clips boxes to valid image boundaries. XML is preferred, with JSON used as a fallback.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def normalize_label(raw_label: str) -> str:
    """Normalize one class label using the fixed alias table."""
    cleaned = str(raw_label).strip()
    return LABEL_ALIASES.get(cleaned.lower(), cleaned)


def read_image_size(image_path: Path) -> Tuple[int, int]:
    """Return image width and height, failing clearly for unreadable files."""
    try:
        with Image.open(image_path) as image:
            return image.size
    except (UnidentifiedImageError, OSError) as error:
        raise ValueError(f"Unreadable image: {image_path}\n{error}") from error


def parse_pascal_voc_xml(
    xml_path: Path,
) -> Tuple[Optional[Tuple[int, int]], List[Dict[str, Any]]]:
    """Parse one Pascal VOC XML annotation."""
    root = ET.parse(xml_path).getroot()
    xml_size = None
    size_node = root.find("size")

    if size_node is not None:
        width_text = size_node.findtext("width")
        height_text = size_node.findtext("height")
        if width_text and height_text:
            xml_size = (int(float(width_text)), int(float(height_text)))

    objects: List[Dict[str, Any]] = []
    for object_node in root.findall("object"):
        raw_name = object_node.findtext("name")
        box_node = object_node.find("bndbox")
        if not raw_name or box_node is None:
            continue

        coordinates = [
            box_node.findtext("xmin"),
            box_node.findtext("ymin"),
            box_node.findtext("xmax"),
            box_node.findtext("ymax"),
        ]
        if any(value is None for value in coordinates):
            continue

        try:
            x1, y1, x2, y2 = [float(value) for value in coordinates]
        except (TypeError, ValueError):
            continue

        if x2 <= x1 + 1 or y2 <= y1 + 1:
            continue

        objects.append({
            "label": normalize_label(raw_name),
            "bbox_xyxy": [x1, y1, x2, y2],
        })

    return xml_size, objects


def extract_via_label(attributes: Dict[str, Any]) -> Optional[str]:
    """Extract a label from common VIA region-attribute layouts."""
    preferred_keys = [
        "label", "class", "class_name", "name", "species", "type", "weed", "category"
    ]

    for key in preferred_keys:
        value = attributes.get(key)
        if isinstance(value, str) and value.strip():
            return normalize_label(value)
        if isinstance(value, dict):
            for nested_key, selected in value.items():
                if selected in {True, 1, "1", "true", "True", "yes", "Yes"}:
                    return normalize_label(nested_key)
            if len(value) == 1:
                return normalize_label(next(iter(value.keys())))

    for value in attributes.values():
        if isinstance(value, str) and value.strip():
            return normalize_label(value)

    return None


def parse_json_annotation(json_path: Path) -> List[Dict[str, Any]]:
    """Parse LabelMe-style or VIA-style JSON annotations."""
    data = json.loads(json_path.read_text(encoding="utf-8"))
    objects: List[Dict[str, Any]] = []

    # LabelMe
    if isinstance(data, dict) and isinstance(data.get("shapes"), list):
        for shape in data["shapes"]:
            label = normalize_label(shape.get("label", ""))
            points = shape.get("points", [])
            if not label or len(points) < 2:
                continue

            xs = [float(point[0]) for point in points]
            ys = [float(point[1]) for point in points]
            x1, x2 = min(xs), max(xs)
            y1, y2 = min(ys), max(ys)

            if x2 > x1 + 1 and y2 > y1 + 1:
                objects.append({"label": label, "bbox_xyxy": [x1, y1, x2, y2]})
        return objects

    # VIA
    metadata = data.get("_via_img_metadata", data) if isinstance(data, dict) else {}
    if not isinstance(metadata, dict):
        return objects

    for entry in metadata.values():
        if not isinstance(entry, dict):
            continue

        regions = entry.get("regions", [])
        if isinstance(regions, dict):
            regions = [regions] if "shape_attributes" in regions else list(regions.values())
        if not isinstance(regions, list):
            continue

        for region in regions:
            shape = region.get("shape_attributes", {}) or {}
            attributes = region.get("region_attributes", {}) or {}
            if shape.get("name") != "rect":
                continue

            values = [shape.get("x"), shape.get("y"), shape.get("width"), shape.get("height")]
            if any(value is None for value in values):
                continue

            label = extract_via_label(attributes)
            if not label:
                continue

            try:
                x, y, width, height = [float(value) for value in values]
            except (TypeError, ValueError):
                continue

            if width > 1 and height > 1:
                objects.append({
                    "label": label,
                    "bbox_xyxy": [x, y, x + width, y + height],
                })

    return objects


def clip_box(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Optional[List[float]]:
    """Clip one xyxy box to image boundaries."""
    x1, y1, x2, y2 = box
    x1 = max(0.0, min(float(x1), image_width - 1.0))
    y1 = max(0.0, min(float(y1), image_height - 1.0))
    x2 = max(0.0, min(float(x2), float(image_width)))
    y2 = max(0.0, min(float(y2), float(image_height)))

    if x2 <= x1 + 1 or y2 <= y1 + 1:
        return None
    return [x1, y1, x2, y2]

## Module 3: Build and audit source and target records

This module scans both yearly datasets, keeps valid annotations for the eight study classes, checks class coverage, saves compact audit summaries, and plots one source versus target class distribution figure.


In [ ]:
def build_clean_records(
    source_root: Path,
) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    """Build clean detection records from one source dataset folder."""
    source_root = Path(source_root)
    if not source_root.exists():
        raise FileNotFoundError(f"Source dataset not found: {source_root.name}")

    image_root = next(
        (p for p in [source_root / "JPEGImages", source_root / "images", source_root] if p.exists()),
        None,
    )
    if image_root is None:
        raise FileNotFoundError(f"No image root found below: {source_root.name}")

    image_paths = sorted(
        p for p in image_root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        raise FileNotFoundError(f"No images found below: {source_root.name}")

    image_names = [p.name for p in image_paths]
    if len(image_names) != len(set(image_names)):
        raise ValueError(
            f"Duplicate image filenames found in {source_root.name}; "
            "flat YOLO split folders require unique filenames within each dataset."
        )

    xml_by_stem = {p.stem: p for p in source_root.rglob("*.xml")}
    json_by_stem = {p.stem: p for p in source_root.rglob("*.json")}

    records: List[Dict[str, Any]] = []
    stats = {
        "images_found": len(image_paths),
        "records_kept": 0,
        "missing_annotation": 0,
        "empty_after_filtering": 0,
        "unreadable_images": 0,
        "invalid_boxes_removed": 0,
        "non_target_objects_removed": 0,
        "used_xml": 0,
        "used_json": 0,
    }

    for image_path in image_paths:
        xml_path = xml_by_stem.get(image_path.stem)
        json_path = json_by_stem.get(image_path.stem)
        annotation_size = None
        objects: List[Dict[str, Any]] = []
        source_used = None

        try:
            if xml_path is not None:
                annotation_size, xml_objects = parse_pascal_voc_xml(xml_path)
                if xml_objects:
                    objects = xml_objects
                    source_used = "xml"

            # Fallback to JSON when XML is absent or produced no usable objects.
            if not objects and json_path is not None:
                objects = parse_json_annotation(json_path)
                if objects:
                    source_used = "json"
                    annotation_size = None

            if not objects:
                stats["missing_annotation"] += 1
                continue

            if annotation_size is None:
                width, height = read_image_size(image_path)
            else:
                width, height = annotation_size

        except Exception as error:
            stats["unreadable_images"] += 1
            print(f"Skipping {image_path.name}: {error}")
            continue

        clean_objects: List[Dict[str, Any]] = []
        for obj in objects:
            label = normalize_label(obj["label"])
            if label not in TARGET_CLASS_SET:
                stats["non_target_objects_removed"] += 1
                continue

            clipped = clip_box(obj["bbox_xyxy"], width, height)
            if clipped is None:
                stats["invalid_boxes_removed"] += 1
                continue

            clean_objects.append({"label": label, "bbox_xyxy": clipped})

        if not clean_objects:
            stats["empty_after_filtering"] += 1
            continue

        stats[f"used_{source_used}"] += 1
        records.append({
            "img_path": str(image_path.resolve()),
            "file_name": image_path.name,
            "width": int(width),
            "height": int(height),
            "objects": clean_objects,
        })

    records.sort(key=lambda row: row["file_name"])
    stats["records_kept"] = len(records)

    if not records:
        raise RuntimeError(f"No usable records remained after filtering: {source_root.name}")

    return records, stats


def audit_records(
    records_to_audit: List[Dict[str, Any]],
    class_names: List[str],
) -> Dict[str, Any]:
    """Count images, boxes, and class occurrence statistics."""
    box_counts = {name: 0 for name in class_names}
    image_counts = {name: 0 for name in class_names}

    for record in records_to_audit:
        labels = [obj["label"] for obj in record["objects"]]
        for label in labels:
            box_counts[label] += 1
        for label in set(labels):
            image_counts[label] += 1

    return {
        "num_images": len(records_to_audit),
        "num_boxes": int(sum(box_counts.values())),
        "box_count_per_class": box_counts,
        "image_count_per_class": image_counts,
        "missing_classes": [name for name in class_names if box_counts[name] == 0],
    }


def plot_source_target_box_counts(
    source_audit: Dict[str, Any],
    target_audit: Dict[str, Any],
    output_root: Path,
) -> None:
    """Plot one grouped comparison of full-dataset box counts by class."""
    output_root.mkdir(parents=True, exist_ok=True)
    x = np.arange(len(CLASS_NAMES))
    width = 0.38

    plt.figure(figsize=(11, 5))
    plt.bar(
        x - width / 2,
        [source_audit["box_count_per_class"][name] for name in CLASS_NAMES],
        width,
        label=f"{TRAIN_DATASET_TAG} source",
    )
    plt.bar(
        x + width / 2,
        [target_audit["box_count_per_class"][name] for name in CLASS_NAMES],
        width,
        label=f"{TEST_DATASET_TAG} target",
    )
    plt.xticks(x, CLASS_NAMES, rotation=45, ha="right")
    plt.ylabel("Number of annotated boxes")
    plt.title(f"{CROSS_YEAR_TAG}: source and target class counts")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_root / "source_target_box_counts.png", dpi=180, bbox_inches="tight")
    plt.show()


SOURCE_RECORDS, SOURCE_SCAN_STATS = build_clean_records(TRAIN_SOURCE_ROOT)
TARGET_RECORDS, TARGET_SCAN_STATS = build_clean_records(TEST_TARGET_ROOT)

SOURCE_AUDIT = audit_records(SOURCE_RECORDS, CLASS_NAMES)
TARGET_AUDIT = audit_records(TARGET_RECORDS, CLASS_NAMES)

if SOURCE_AUDIT["missing_classes"]:
    raise RuntimeError(f"Source dataset is missing target classes: {SOURCE_AUDIT['missing_classes']}")
if TARGET_AUDIT["missing_classes"]:
    raise RuntimeError(f"Target dataset is missing target classes: {TARGET_AUDIT['missing_classes']}")

(STATS_ROOT / f"{TRAIN_DATASET_TAG}_source_scan_summary.json").write_text(
    json.dumps(SOURCE_SCAN_STATS, indent=2), encoding="utf-8"
)
(STATS_ROOT / f"{TEST_DATASET_TAG}_target_scan_summary.json").write_text(
    json.dumps(TARGET_SCAN_STATS, indent=2), encoding="utf-8"
)
(STATS_ROOT / f"{TRAIN_DATASET_TAG}_source_class_audit.json").write_text(
    json.dumps(SOURCE_AUDIT, indent=2), encoding="utf-8"
)
(STATS_ROOT / f"{TEST_DATASET_TAG}_target_class_audit.json").write_text(
    json.dumps(TARGET_AUDIT, indent=2), encoding="utf-8"
)

print("\nSOURCE DATASET AUDIT")
print(json.dumps({"scan": SOURCE_SCAN_STATS, "audit": SOURCE_AUDIT}, indent=2))
print("\nTARGET DATASET AUDIT")
print(json.dumps({"scan": TARGET_SCAN_STATS, "audit": TARGET_AUDIT}, indent=2))

plot_source_target_box_counts(SOURCE_AUDIT, TARGET_AUDIT, RUN_PLOTS_ROOT)


## Module 4: Deterministic cross year splits and source subsampling

This module creates reproducible 70/15/15 splits for both years. The effective experiment uses source train, source validation, and target test. If Year2021 is the source year, only its training split is stratified subsampled to match the target training split size. Leakage and class coverage are checked before export.


In [ ]:
def split_records_deterministically(
    all_records: List[Dict[str, Any]],
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    seed: int,
) -> Dict[str, List[Dict[str, Any]]]:
    """Create one reproducible 70/15/15 split."""
    if abs(train_ratio + valid_ratio + test_ratio - 1.0) > 1e-9:
        raise ValueError("Split ratios must sum to 1.0.")

    shuffled = list(all_records)
    random.Random(seed).shuffle(shuffled)

    total = len(shuffled)
    n_train = int(total * train_ratio)
    n_valid = int(total * valid_ratio)

    return {
        "train": shuffled[:n_train],
        "valid": shuffled[n_train:n_train + n_valid],
        "test": shuffled[n_train + n_valid:],
    }


def get_record_primary_label(record: Dict[str, Any]) -> str:
    """
    Assign one primary class to an image for stratified subsampling.

    For multi-class images, use the most frequent class.
    Ties follow the fixed CLASS_NAMES order.
    """
    labels = [obj["label"] for obj in record.get("objects", [])]
    if not labels:
        return "unknown"

    counts = Counter(labels)
    return max(
        CLASS_NAMES,
        key=lambda class_name: (
            counts.get(class_name, 0),
            -CLASS_NAMES.index(class_name),
        ),
    )


def _exact_stratified_allocation(
    groups: Dict[str, List[Dict[str, Any]]],
    target_size: int,
) -> Dict[str, int]:
    """
    Allocate an exact target size approximately in proportion to group sizes,
    while retaining at least one record per represented group when possible.
    """
    labels = [label for label, group in groups.items() if group]
    total = sum(len(groups[label]) for label in labels)

    if target_size < 0 or target_size > total:
        raise ValueError("target_size must be between 0 and the number of available records.")
    if target_size == 0:
        return {label: 0 for label in labels}

    raw = {label: target_size * len(groups[label]) / total for label in labels}
    allocation = {label: min(int(math.floor(raw[label])), len(groups[label])) for label in labels}

    # Preserve represented classes where the requested sample is large enough.
    if target_size >= len(labels):
        for label in labels:
            if allocation[label] == 0:
                allocation[label] = 1

    # If minimum-one adjustments overshoot, remove from the most over-allocated
    # groups while keeping at least one when that rule is active.
    min_allowed = 1 if target_size >= len(labels) else 0
    while sum(allocation.values()) > target_size:
        reducible = [
            label for label in labels
            if allocation[label] > min_allowed
        ]
        if not reducible:
            raise RuntimeError("Could not reduce stratified allocation to the exact target size.")
        label = max(
            reducible,
            key=lambda name: (allocation[name] - raw[name], allocation[name], len(groups[name])),
        )
        allocation[label] -= 1

    # Fill remaining slots using largest remainder, respecting group capacity.
    while sum(allocation.values()) < target_size:
        candidates = [
            label for label in labels
            if allocation[label] < len(groups[label])
        ]
        if not candidates:
            raise RuntimeError("Could not fill stratified allocation to the exact target size.")
        label = max(
            candidates,
            key=lambda name: (raw[name] - allocation[name], len(groups[name])),
        )
        allocation[label] += 1

    if sum(allocation.values()) != target_size:
        raise RuntimeError("Stratified allocation did not reach the requested target size.")

    return allocation


def stratified_subsample_records(
    records: List[Dict[str, Any]],
    target_size: int,
    seed: int,
) -> List[Dict[str, Any]]:
    """Stratified image-level subsampling using the primary class of each image."""
    if target_size >= len(records):
        return list(records)

    rng = random.Random(seed)
    groups: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

    for record in records:
        groups[get_record_primary_label(record)].append(record)

    for group in groups.values():
        rng.shuffle(group)

    allocation = _exact_stratified_allocation(groups, target_size)

    sampled: List[Dict[str, Any]] = []
    for label, group in groups.items():
        sampled.extend(group[:allocation.get(label, 0)])

    rng.shuffle(sampled)

    if len(sampled) != target_size:
        raise RuntimeError(
            f"Subsampling produced {len(sampled)} records; expected exactly {target_size}."
        )
    return sampled


def maybe_subsample_source_train(
    source_train_records: List[Dict[str, Any]],
    target_splits: Dict[str, List[Dict[str, Any]]],
    seed: int,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Apply the established subsampling rule only when Year2021 is the source year."""
    should_subsample = (
        SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE
        and TRAIN_YEAR == "2021"
    )

    info = {
        "enabled": bool(should_subsample),
        "source_year": TRAIN_YEAR,
        "target_year": TEST_YEAR,
        "mode": SUBSAMPLE_TARGET_MODE,
        "subsample_seed": None,
        "original_source_train_count": len(source_train_records),
        "target_size": None,
        "final_source_train_count": len(source_train_records),
    }

    if not should_subsample:
        return list(source_train_records), info

    if SUBSAMPLE_TARGET_MODE != "target_train_size":
        raise ValueError("This notebook freezes SUBSAMPLE_TARGET_MODE='target_train_size'.")

    target_size = len(target_splits["train"])
    subsample_seed = int(seed + SUBSAMPLE_SEED_OFFSET)

    if target_size > len(source_train_records):
        # Do not upsample. This guard makes the rule explicit and auditable.
        print(
            "WARNING: target train size exceeds Year2021 source train size; "
            "subsampling is skipped because the protocol does not upsample."
        )
        info["target_size"] = int(target_size)
        info["subsample_seed"] = subsample_seed
        return list(source_train_records), info

    sampled = stratified_subsample_records(
        records=source_train_records,
        target_size=target_size,
        seed=subsample_seed,
    )

    info["target_size"] = int(target_size)
    info["subsample_seed"] = subsample_seed
    info["final_source_train_count"] = len(sampled)
    return sampled, info


def _path_set(records: List[Dict[str, Any]]) -> set:
    return {str(Path(record["img_path"]).resolve()) for record in records}


def verify_cross_split_disjointness(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
) -> None:
    """Fail if any physical image path occurs in more than one effective split."""
    train_paths = _path_set(cross_year_splits["train"])
    valid_paths = _path_set(cross_year_splits["valid"])
    test_paths = _path_set(cross_year_splits["test"])

    overlaps = {
        "train_valid": train_paths & valid_paths,
        "train_test": train_paths & test_paths,
        "valid_test": valid_paths & test_paths,
    }
    bad = {name: paths for name, paths in overlaps.items() if paths}
    if bad:
        raise RuntimeError(
            "Cross-year split leakage detected: "
            + ", ".join(f"{name}={len(paths)}" for name, paths in bad.items())
        )


def audit_cross_year_splits(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
    fail_on_missing: bool = True,
) -> Dict[str, Any]:
    """Audit effective train/valid/test class coverage."""
    report: Dict[str, Any] = {}
    for split_name, rows in cross_year_splits.items():
        split_audit = audit_records(rows, CLASS_NAMES)
        report[split_name] = split_audit
        print(
            f"{split_name:8s} images={split_audit['num_images']:5d} "
            f"boxes={split_audit['num_boxes']:6d} "
            f"missing={split_audit['missing_classes']}"
        )
        if fail_on_missing and split_audit["missing_classes"]:
            raise RuntimeError(
                f"Effective split '{split_name}' is missing classes: "
                f"{split_audit['missing_classes']}"
            )
    return report


def make_cross_year_splits_for_seed(
    seed: int,
) -> Tuple[
    Dict[str, List[Dict[str, Any]]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, Any],
]:
    """
    Build source splits, target splits, and the effective cross-year layout.

    Effective YOLO layout:
        train = source train (subsampled only if source is Year2021)
        valid = source validation
        test  = target test
    """
    source_splits = split_records_deterministically(
        SOURCE_RECORDS, TRAIN_RATIO, VALID_RATIO, TEST_RATIO, seed
    )
    target_splits = split_records_deterministically(
        TARGET_RECORDS, TRAIN_RATIO, VALID_RATIO, TEST_RATIO, seed
    )

    effective_source_train, subsample_info = maybe_subsample_source_train(
        source_train_records=source_splits["train"],
        target_splits=target_splits,
        seed=seed,
    )

    cross_year_splits = {
        "train": effective_source_train,
        "valid": source_splits["valid"],
        "test": target_splits["test"],
    }

    verify_cross_split_disjointness(cross_year_splits)
    coverage = audit_cross_year_splits(cross_year_splits, fail_on_missing=True)

    split_info = {
        "seed": int(seed),
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "source_dataset": TRAIN_DATASET_TAG,
        "target_dataset": TEST_DATASET_TAG,
        "ratios": {
            "train": TRAIN_RATIO,
            "valid": VALID_RATIO,
            "test": TEST_RATIO,
        },
        "source_counts_original": {
            name: len(rows) for name, rows in source_splits.items()
        },
        "target_counts_original": {
            name: len(rows) for name, rows in target_splits.items()
        },
        "effective_cross_year_counts": {
            name: len(rows) for name, rows in cross_year_splits.items()
        },
        "subsample_info": subsample_info,
        "class_coverage": coverage,
    }

    return cross_year_splits, source_splits, target_splits, split_info


# Preview and audit the optimization split before any GPU training.
PREVIEW_CROSS_SPLITS, PREVIEW_SOURCE_SPLITS, PREVIEW_TARGET_SPLITS, PREVIEW_SPLIT_INFO = (
    make_cross_year_splits_for_seed(OPTIMIZATION_SPLIT_SEED)
)

print("\nCROSS-YEAR OPTIMIZATION SPLIT PREVIEW")
print("=" * 96)
print(json.dumps(PREVIEW_SPLIT_INFO, indent=2))

## Module 5: Export the cross year YOLO dataset and verify integrity

This module creates the temporary YOLO folder layout, writes label files and `dataset.yaml`, saves filename only split manifests, and verifies that every exported split has the expected number of images and labels.


In [ ]:
def remove_file_or_link(path: Path) -> None:
    """Remove a regular file or symlink, including broken symlinks."""
    if path.is_symlink() or path.is_file():
        path.unlink()


def clear_split_directory(directory: Path) -> None:
    """Clear files, symlinks, and old disk-cache .npy files from one split folder."""
    directory.mkdir(parents=True, exist_ok=True)
    for path in directory.iterdir():
        if path.is_file() or path.is_symlink():
            path.unlink()


def place_image(source: Path, destination: Path, mode: str) -> str:
    """Place one image using symlink -> hardlink -> copy fallback."""
    if mode not in {"symlink", "hardlink", "copy"}:
        raise ValueError("mode must be 'symlink', 'hardlink', or 'copy'.")

    source = source.resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    remove_file_or_link(destination)

    if mode == "symlink":
        try:
            destination.symlink_to(source)
            return "symlinked"
        except OSError:
            pass

    if mode in {"symlink", "hardlink"}:
        try:
            os.link(source, destination)
            return "hardlinked"
        except OSError:
            pass

    shutil.copy2(source, destination)
    return "copied"


def xyxy_to_yolo(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Tuple[float, float, float, float]:
    """Convert absolute xyxy coordinates to normalized YOLO xywh."""
    x1, y1, x2, y2 = box
    values = (
        ((x1 + x2) / 2.0) / image_width,
        ((y1 + y2) / 2.0) / image_height,
        (x2 - x1) / image_width,
        (y2 - y1) / image_height,
    )
    return tuple(max(0.0, min(1.0, float(value))) for value in values)


def write_yolo_label(
    record: Dict[str, Any],
    output_path: Path,
    class_to_id: Dict[str, int],
) -> None:
    """Write one YOLO label file."""
    lines: List[str] = []
    for obj in record["objects"]:
        class_id = class_to_id[obj["label"]]
        xc, yc, bw, bh = xyxy_to_yolo(
            obj["bbox_xyxy"], record["width"], record["height"]
        )
        lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_dataset_yaml(
    dataset_root: Path,
    class_names: List[str],
) -> Path:
    """Write an Ultralytics dataset.yaml for the effective cross-year layout."""
    yaml_path = dataset_root / "dataset.yaml"
    lines = [
        f"path: {dataset_root.resolve().as_posix()}",
        "train: images/train",
        "val: images/valid",
        "test: images/test",
        f"nc: {len(class_names)}",
        "names:",
    ]
    lines.extend(f"  {idx}: {name}" for idx, name in enumerate(class_names))
    yaml_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return yaml_path


def save_split_manifest(
    split_records: Dict[str, List[Dict[str, Any]]],
    output_json: Path,
    seed: int,
    dataset_tag: str,
    role: str,
    extra_info: Optional[Dict[str, Any]] = None,
) -> None:
    """Save filenames and counts for reproducibility."""
    output_json.parent.mkdir(parents=True, exist_ok=True)
    manifest = {
        "dataset_tag": dataset_tag,
        "role": role,
        "seed": int(seed),
        "counts": {name: len(rows) for name, rows in split_records.items()},
        "files": {
            name: [record["file_name"] for record in rows]
            for name, rows in split_records.items()
        },
    }
    if extra_info is not None:
        manifest["extra_info"] = extra_info
    output_json.write_text(json.dumps(manifest, indent=2), encoding="utf-8")


def count_image_files(directory: Path) -> int:
    """Count only actual image files/symlinks, excluding disk-cache .npy files."""
    return sum(
        1 for path in directory.iterdir()
        if (path.is_file() or path.is_symlink())
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def count_label_files(directory: Path) -> int:
    return sum(
        1 for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() == ".txt"
    )


def check_yolo_dataset_integrity(
    dataset_root: Path,
    expected_counts: Dict[str, int],
) -> None:
    """Verify expected image and label counts for every effective split."""
    for split_name in ("train", "valid", "test"):
        image_dir = dataset_root / "images" / split_name
        label_dir = dataset_root / "labels" / split_name
        image_count = count_image_files(image_dir)
        label_count = count_label_files(label_dir)
        expected = int(expected_counts[split_name])

        print(
            f"{split_name:8s} images={image_count:5d} "
            f"labels={label_count:5d} expected={expected:5d}"
        )
        if image_count != expected or label_count != expected:
            raise RuntimeError(
                f"YOLO integrity failure in '{split_name}': "
                f"images={image_count}, labels={label_count}, expected={expected}."
            )


def prepare_cross_year_yolo_dataset(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
    source_splits: Dict[str, List[Dict[str, Any]]],
    target_splits: Dict[str, List[Dict[str, Any]]],
    split_info: Dict[str, Any],
    output_root: Path,
    manifest_root: Path,
    image_mode: str,
    seed: int,
) -> Tuple[Path, Dict[str, Any]]:
    """Build and verify one effective cross-year YOLO dataset."""
    output_root = Path(output_root)
    manifest_root = Path(manifest_root)
    class_to_id = {name: idx for idx, name in enumerate(CLASS_NAMES)}

    # Recheck leakage and class coverage immediately before export.
    verify_cross_split_disjointness(cross_year_splits)
    coverage = audit_cross_year_splits(cross_year_splits, fail_on_missing=True)

    image_dirs = {
        name: output_root / "images" / name for name in ("train", "valid", "test")
    }
    label_dirs = {
        name: output_root / "labels" / name for name in ("train", "valid", "test")
    }

    for directory in [*image_dirs.values(), *label_dirs.values()]:
        clear_split_directory(directory)

    placement_counts = {"symlinked": 0, "hardlinked": 0, "copied": 0}

    for split_name, rows in cross_year_splits.items():
        for record in rows:
            source = Path(record["img_path"])
            destination = image_dirs[split_name] / record["file_name"]
            action = place_image(source, destination, image_mode)
            placement_counts[action] += 1

            label_path = label_dirs[split_name] / f"{Path(record['file_name']).stem}.txt"
            write_yolo_label(record, label_path, class_to_id)

    yaml_path = write_dataset_yaml(output_root, CLASS_NAMES)

    save_split_manifest(
        source_splits,
        manifest_root / f"{TRAIN_DATASET_TAG}_source_seed{seed}.json",
        seed,
        TRAIN_DATASET_TAG,
        "source",
    )
    save_split_manifest(
        target_splits,
        manifest_root / f"{TEST_DATASET_TAG}_target_seed{seed}.json",
        seed,
        TEST_DATASET_TAG,
        "target",
    )
    save_split_manifest(
        cross_year_splits,
        manifest_root / f"cross_{CROSS_YEAR_TAG}_seed{seed}.json",
        seed,
        CROSS_YEAR_TAG,
        "effective_cross_year",
        extra_info=split_info,
    )

    expected_counts = {
        name: len(rows) for name, rows in cross_year_splits.items()
    }
    check_yolo_dataset_integrity(output_root, expected_counts)

    layout_manifest = {
        "cross_year_tag": CROSS_YEAR_TAG,
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "seed": int(seed),
        "dataset_yaml": relative_output_path(yaml_path, WORKSPACE_ROOT),
        "output_root": relative_output_path(output_root, WORKSPACE_ROOT),
        "placement_counts": placement_counts,
        "expected_counts": expected_counts,
        "split_info": split_info,
    }
    manifest_root.mkdir(parents=True, exist_ok=True)
    (manifest_root / "yolo_layout_manifest.json").write_text(
        json.dumps(layout_manifest, indent=2), encoding="utf-8"
    )

    print("Dataset YAML:", relative_output_path(yaml_path, WORKSPACE_ROOT))
    print("Image placement:", placement_counts)
    return yaml_path, coverage


## Module 6: Metrics, checkpoint alignment, and training helper

This module extracts YOLO detection metrics, checks that `best.pt` selection is aligned with mAP50-95, and defines the shared training routine used by both HPO trials and final seeded runs.


In [ ]:
def extract_box_metrics(
    metrics: Any,
    expected_class_names: List[str],
) -> Dict[str, Any]:
    """Extract detection metrics; optimization fitness is mAP50-95 only."""
    box = getattr(metrics, "box", None)
    if box is None:
        raise RuntimeError("Ultralytics returned no detection box metrics.")

    precision = float(box.mp)
    recall = float(box.mr)
    map50 = float(box.map50)
    map50_95 = float(box.map)

    names_obj = getattr(metrics, "names", {})
    if isinstance(names_obj, dict):
        class_name_lookup = {int(k): str(v) for k, v in names_obj.items()}
    else:
        class_name_lookup = {idx: str(name) for idx, name in enumerate(names_obj)}

    per_class_rows: List[Dict[str, Any]] = []

    if hasattr(box, "ap") and hasattr(box, "ap_class_index"):
        ap_values = np.asarray(box.ap, dtype=float).reshape(-1)
        class_ids = np.asarray(box.ap_class_index, dtype=int).reshape(-1)

        for class_id, ap_value in zip(class_ids.tolist(), ap_values.tolist()):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(int(class_id), str(class_id)),
                "map50_95": float(ap_value),
            })
    else:
        maps = [float(value) for value in box.maps]
        for class_id, ap_value in enumerate(maps):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(int(class_id), str(class_id)),
                "map50_95": float(ap_value),
            })

    represented_ids = {int(row["class_id"]) for row in per_class_rows}
    expected_ids = set(range(len(expected_class_names)))
    missing_ids = sorted(expected_ids - represented_ids)

    if missing_ids:
        missing_names = [expected_class_names[idx] for idx in missing_ids]
        raise RuntimeError(
            "Validation/test metrics are missing represented classes: "
            f"{missing_names}. Check the split coverage."
        )

    sorted_aps = sorted(float(row["map50_95"]) for row in per_class_rows)
    bottom3 = float(mean(sorted_aps[:3])) if len(sorted_aps) >= 3 else float(mean(sorted_aps))

    return {
        "precision": precision,
        "recall": recall,
        "map50": map50,
        "map50_95": map50_95,
        "fitness": map50_95,
        "bottom3_mean_map50_95": bottom3,
        "per_class_map50_95": per_class_rows,
    }


def read_epoch_map50_95_values(results_csv: Path) -> Tuple[str, List[float]]:
    """Read epoch-level mAP50-95 values from an Ultralytics results.csv file."""
    results_csv = Path(results_csv)
    if not results_csv.exists():
        raise FileNotFoundError(f"Training results.csv not found: {results_csv}")

    with results_csv.open("r", newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        fieldnames = reader.fieldnames or []

    candidate_columns = [name for name in fieldnames if "mAP50-95" in name]
    if not candidate_columns:
        raise RuntimeError(
            f"Could not find an mAP50-95 column in {results_csv}. "
            f"Available columns: {fieldnames}"
        )

    metric_column = next(
        (name for name in candidate_columns if "metrics/" in name),
        candidate_columns[0],
    )

    values: List[float] = []
    for row in rows:
        raw_value = row.get(metric_column, "")
        try:
            value = float(raw_value)
        except (TypeError, ValueError):
            continue
        if math.isfinite(value):
            values.append(value)

    if not values:
        raise RuntimeError(f"No numeric values found in results column: {metric_column}")

    return metric_column, values


def verify_ultralytics_fitness_alignment(
    results_csv: Path,
    trainer_best_fitness: Optional[float],
    tolerance: float,
    strict: bool,
) -> Dict[str, Any]:
    """Verify that trainer.best_fitness agrees with maximum epoch mAP50-95."""
    metric_column, map95_values = read_epoch_map50_95_values(results_csv)
    max_epoch_map95 = float(max(map95_values))

    if trainer_best_fitness is None or not math.isfinite(float(trainer_best_fitness)):
        report = {
            "status": "UNVERIFIED",
            "metric_column": metric_column,
            "max_epoch_map50_95": max_epoch_map95,
            "trainer_best_fitness": None,
            "absolute_difference": None,
            "tolerance": float(tolerance),
        }
        if strict:
            raise RuntimeError(
                "Could not read Ultralytics trainer.best_fitness, so checkpoint-fitness "
                "alignment could not be verified."
            )
        return report

    trainer_best_fitness = float(trainer_best_fitness)
    difference = abs(trainer_best_fitness - max_epoch_map95)
    status = "PASS" if difference <= tolerance else "FAIL"

    report = {
        "status": status,
        "metric_column": metric_column,
        "max_epoch_map50_95": max_epoch_map95,
        "trainer_best_fitness": trainer_best_fitness,
        "absolute_difference": float(difference),
        "tolerance": float(tolerance),
    }

    if strict and status != "PASS":
        raise RuntimeError(
            "Ultralytics checkpoint fitness does not align with mAP50-95. "
            f"best_fitness={trainer_best_fitness:.6f}, "
            f"max epoch mAP50-95={max_epoch_map95:.6f}, "
            f"difference={difference:.6f}."
        )

    return report


def train_and_validate_configuration(
    *,
    learning_rate: float,
    lrf: float,
    momentum: float,
    weight_decay: float,
    scale: float,
    dataset_yaml: Path,
    project_dir: Path,
    model_name: str,
    epochs: int,
    image_size: int,
    batch_size: int,
    patience: int,
    workers: int,
    seed: int,
    warmup_epochs: float,
    close_mosaic: int,
    fixed_training_args: Dict[str, Any],
    expected_class_names: List[str],
    strict_fitness_alignment_check: bool,
    fitness_alignment_tolerance: float,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Train from scratch, verify checkpoint alignment, then validate best.pt."""
    dataset_yaml = Path(dataset_yaml)
    project_dir = Path(project_dir)

    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")
    if learning_rate <= 0 or lrf <= 0:
        raise ValueError("learning_rate and lrf must be greater than 0.")
    if not 0.0 < momentum < 1.0:
        raise ValueError("momentum must be between 0 and 1.")
    if weight_decay < 0:
        raise ValueError("weight_decay cannot be negative.")
    if not 0.0 <= scale <= 1.0:
        raise ValueError("scale must be between 0 and 1.")
    if epochs < 1 or batch_size < 1 or workers < 0:
        raise ValueError("Invalid epochs, batch_size, or workers value.")
    if warmup_epochs < 0 or close_mosaic < 0:
        raise ValueError("warmup_epochs and close_mosaic must be non-negative.")

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"
    using_cuda = str(device) != "cpu" and torch.cuda.is_available()

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    trial_id = uuid.uuid4().hex[:8]
    run_name = f"CY{TRAIN_YEAR}{TEST_YEAR}_s{seed}_e{epochs}_{trial_id}"
    project_dir.mkdir(parents=True, exist_ok=True)

    hyperparameters = {
        "learning_rate": float(learning_rate),
        "lrf": float(lrf),
        "momentum": float(momentum),
        "weight_decay": float(weight_decay),
        "scale": float(scale),
        "epochs": int(epochs),
        "warmup_epochs": float(warmup_epochs),
        "close_mosaic": int(close_mosaic),
        "seed": int(seed),
        "patience": int(patience),
    }

    print("\n" + "=" * 88)
    print("START TRAINING:", run_name)
    print(json.dumps(hyperparameters, indent=2))
    print("=" * 88)

    train_kwargs = dict(fixed_training_args)
    train_kwargs.update({
        "data": str(dataset_yaml.resolve()),
        "project": str(project_dir.resolve()),
        "name": run_name,
        "epochs": int(epochs),
        "imgsz": int(image_size),
        "batch": int(batch_size),
        "patience": int(patience),
        "workers": int(workers),
        "device": device,
        "seed": int(seed),
        "deterministic": True,
        "pretrained": True,
        "val": True,
        "save": True,
        "save_period": -1,
        "plots": False,
        "verbose": True,
        "amp": bool(using_cuda),
        "lr0": float(learning_rate),
        "lrf": float(lrf),
        "momentum": float(momentum),
        "weight_decay": float(weight_decay),
        "scale": float(scale),
        "warmup_epochs": float(warmup_epochs),
        "close_mosaic": int(close_mosaic),
    })

    model = YOLO(model_name)
    train_start = time.perf_counter()
    train_result = model.train(**train_kwargs)
    training_seconds = time.perf_counter() - train_start

    train_save_dir = getattr(train_result, "save_dir", None)
    if train_save_dir is None and getattr(model, "trainer", None) is not None:
        train_save_dir = getattr(model.trainer, "save_dir", None)
    if train_save_dir is None:
        raise RuntimeError("Training finished but save_dir could not be resolved.")

    train_save_dir = Path(train_save_dir)
    best_checkpoint = train_save_dir / "weights" / "best.pt"
    results_csv = train_save_dir / "results.csv"

    if not best_checkpoint.exists():
        raise FileNotFoundError(f"best.pt not found: {best_checkpoint}")
    if not results_csv.exists():
        raise FileNotFoundError(f"results.csv not found: {results_csv}")

    trainer_best_fitness = None
    trainer = getattr(model, "trainer", None)
    if trainer is not None:
        raw_best_fitness = getattr(trainer, "best_fitness", None)
        if raw_best_fitness is not None:
            trainer_best_fitness = float(raw_best_fitness)

    alignment_report = verify_ultralytics_fitness_alignment(
        results_csv=results_csv,
        trainer_best_fitness=trainer_best_fitness,
        tolerance=fitness_alignment_tolerance,
        strict=strict_fitness_alignment_check,
    )

    # Re-evaluate the frozen best checkpoint on the SOURCE validation split.
    best_model = YOLO(str(best_checkpoint))
    val_start = time.perf_counter()
    validation_metrics = best_model.val(
        data=str(dataset_yaml.resolve()),
        split="val",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(project_dir.resolve()),
        name=f"{run_name}_val",
        plots=False,
        save_json=False,
        verbose=False,
    )
    validation_seconds = time.perf_counter() - val_start

    metric_data = extract_box_metrics(validation_metrics, expected_class_names)

    peak_gpu_memory_gb = None
    if using_cuda:
        peak_gpu_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    result = {
        "status": "PASS",
        "trial_id": trial_id,
        "run_name": run_name,
        "cross_year_tag": CROSS_YEAR_TAG,
        "hyperparameters": hyperparameters,
        "metrics": metric_data,
        "fitness_alignment": alignment_report,
        "timing": {
            "training_seconds": float(training_seconds),
            "validation_seconds": float(validation_seconds),
            "total_seconds": float(training_seconds + validation_seconds),
        },
        "hardware": {
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(0) if using_cuda else None,
            "peak_gpu_memory_gb": (
                float(peak_gpu_memory_gb) if peak_gpu_memory_gb is not None else None
            ),
        },
        "paths": {
            "training_folder": relative_output_path(train_save_dir, WORKSPACE_ROOT),
            "best_checkpoint": relative_output_path(best_checkpoint, WORKSPACE_ROOT),
            "results_csv": relative_output_path(results_csv, WORKSPACE_ROOT),
        },
    }

    (train_save_dir / "trial_summary.json").write_text(
        json.dumps(result, indent=2), encoding="utf-8"
    )

    print(
        f"Completed {run_name}: source-val mAP50-95={metric_data['map50_95']:.6f}, "
        f"P={metric_data['precision']:.6f}, R={metric_data['recall']:.6f}, "
        f"alignment={alignment_report['status']}"
    )
    return result


## Module 7: Random Search, ranking, summaries, and privacy safe outputs

This module builds the deterministic 8 configuration Random Search plan, ranks successful trials by source validation mAP50-95, summarizes final metrics, checks selected parameter positions, and writes privacy safe result files.


In [ ]:
def safe_float(value: Any) -> Optional[float]:
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def save_rows_to_csv_safe(rows: List[Dict[str, Any]], output_csv: Path) -> None:
    """Save heterogeneous dictionaries to CSV with an Excel-open fallback."""
    if not rows:
        return

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    fieldnames: List[str] = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)

    try:
        with output_csv.open("w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
    except PermissionError:
        backup = output_csv.with_name(
            f"{output_csv.stem}_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}{output_csv.suffix}"
        )
        with backup.open("w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        print("CSV was open elsewhere; backup written to:", backup)


def save_json(data: Dict[str, Any], output_json: Path) -> None:
    output_json = Path(output_json)
    output_json.parent.mkdir(parents=True, exist_ok=True)
    output_json.write_text(json.dumps(data, indent=2), encoding="utf-8")


def sample_parameter(
    rng: random.Random,
    low: float,
    high: float,
    distribution: str,
) -> float:
    """Sample one linear- or log-uniform parameter."""
    if not low < high:
        raise ValueError("Search-space low must be < high.")
    if distribution == "log":
        if low <= 0:
            raise ValueError("Log-uniform lower bound must be > 0.")
        return math.exp(rng.uniform(math.log(low), math.log(high)))
    if distribution == "linear":
        return rng.uniform(low, high)
    raise ValueError(f"Unsupported distribution: {distribution}")


def build_random_search_plan(
    n_trials: int,
    random_search_seed: int,
    search_space: Dict[str, Dict[str, Any]],
    parameter_order: List[str],
) -> List[Dict[str, Any]]:
    """Build one deterministic 5D Random Search plan."""
    rng = random.Random(random_search_seed)
    configs: List[Dict[str, Any]] = []

    for trial_index in range(1, n_trials + 1):
        row: Dict[str, Any] = {"trial_index": int(trial_index)}
        for parameter_name in parameter_order:
            spec = search_space[parameter_name]
            row[parameter_name] = float(sample_parameter(
                rng,
                float(spec["low"]),
                float(spec["high"]),
                str(spec["distribution"]),
            ))
        configs.append(row)

    signatures = {
        tuple(round(float(row[name]), 12) for name in parameter_order)
        for row in configs
    }
    if len(signatures) != len(configs):
        raise RuntimeError("Random Search plan contains duplicate configurations.")

    return configs


def rank_by_map50_95(
    rows: List[Dict[str, Any]],
    map95_key: str,
    map50_key: str,
    recall_key: str,
) -> List[Dict[str, Any]]:
    """Rank PASS rows by mAP50-95, then mAP50, recall, and trial index."""
    eligible = [
        row for row in rows
        if str(row.get("status", "")).upper() == "PASS"
        and safe_float(row.get(map95_key)) is not None
    ]

    return sorted(
        eligible,
        key=lambda row: (
            -float(row[map95_key]),
            -float(row[map50_key]),
            -float(row[recall_key]),
            int(row["trial_index"]),
        ),
    )


def summarize_metric_mean_sd(
    rows: List[Dict[str, Any]],
    metric_names: List[str],
) -> List[Dict[str, Any]]:
    """Return mean/sample-SD/min/max summary rows."""
    summary: List[Dict[str, Any]] = []

    for metric_name in metric_names:
        values = [
            value for value in (safe_float(row.get(metric_name)) for row in rows)
            if value is not None
        ]

        if not values:
            summary.append({
                "metric": metric_name, "n": 0,
                "mean": "", "sd": "", "min": "", "max": ""
            })
        elif len(values) == 1:
            summary.append({
                "metric": metric_name, "n": 1,
                "mean": values[0], "sd": "", "min": values[0], "max": values[0]
            })
        else:
            summary.append({
                "metric": metric_name,
                "n": len(values),
                "mean": mean(values),
                "sd": stdev(values),
                "min": min(values),
                "max": max(values),
            })

    return summary


def summarize_per_class_mean_sd(
    per_class_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Summarize target-test per-class mAP50-95 across final seeds."""
    summary: List[Dict[str, Any]] = []

    for class_name in sorted({row["class_name"] for row in per_class_rows}):
        values = [
            float(row["target_test_map50_95"])
            for row in per_class_rows
            if row["class_name"] == class_name
        ]
        summary.append({
            "class_name": class_name,
            "n": len(values),
            "mean_target_test_map50_95": mean(values),
            "sd_target_test_map50_95": "" if len(values) == 1 else stdev(values),
            "min_target_test_map50_95": min(values),
            "max_target_test_map50_95": max(values),
        })

    return summary


def boundary_proximity_report(
    selected_config: Dict[str, Any],
    search_space: Dict[str, Dict[str, Any]],
    parameter_order: List[str],
    boundary_fraction: float = 0.10,
) -> List[Dict[str, Any]]:
    """Flag selected parameters in the lower/upper 10% of their sampling ranges."""
    report: List[Dict[str, Any]] = []

    for name in parameter_order:
        low = float(search_space[name]["low"])
        high = float(search_space[name]["high"])
        value = float(selected_config[name])

        if search_space[name]["distribution"] == "log":
            low_t, high_t, value_t = math.log(low), math.log(high), math.log(value)
        else:
            low_t, high_t, value_t = low, high, value

        fraction = (value_t - low_t) / (high_t - low_t)

        if fraction <= boundary_fraction:
            boundary = "LOW"
        elif fraction >= 1.0 - boundary_fraction:
            boundary = "HIGH"
        else:
            boundary = "INTERIOR"

        report.append({
            "parameter": name,
            "value": value,
            "low": low,
            "high": high,
            "distribution": search_space[name]["distribution"],
            "normalized_position": float(fraction),
            "boundary_flag": boundary,
        })

    return report

## Module 8: Compact diagnostic plots

This module defines two experiment plots: Random Search progress during HPO and the mean plus sample SD of the main target test metrics across the three final seeds. Together with the dataset audit plot, this keeps visualization useful without creating excessive figures.


In [ ]:
def plot_hpo_progress(
    rows: List[Dict[str, Any]],
    output_root: Path,
) -> None:
    """Plot source validation mAP50-95 and the running best by trial."""
    successful = [
        row for row in rows
        if row["status"] == "PASS"
    ]
    if not successful:
        return

    trial_ids = [
        int(row["trial_index"])
        for row in successful
    ]
    scores = [
        float(row["source_val_map50_95"])
        for row in successful
    ]

    running_best: List[float] = []
    current_best = -float("inf")
    for score in scores:
        current_best = max(current_best, score)
        running_best.append(current_best)

    output_root.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(9, 5))
    plt.plot(
        trial_ids,
        scores,
        marker="o",
        label="trial mAP50-95",
    )
    plt.plot(
        trial_ids,
        running_best,
        linestyle="--",
        label="running best",
    )
    plt.xlabel("Random Search trial")
    plt.ylabel("Source validation mAP50-95")
    plt.title(f"{CROSS_YEAR_TAG}: Random Search progress")
    plt.xticks(trial_ids)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        output_root / "random_search_progress.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()


def plot_final_target_test_metrics(
    final_rows: List[Dict[str, Any]],
    output_root: Path,
) -> None:
    """Plot mean and sample SD for the main target test metrics."""
    successful = [
        row for row in final_rows
        if row["status"] == "PASS"
    ]
    if not successful:
        return

    metric_keys = [
        "target_test_precision",
        "target_test_recall",
        "target_test_map50",
        "target_test_map50_95",
    ]
    labels = [
        "Precision",
        "Recall",
        "mAP50",
        "mAP50-95",
    ]

    metric_means: List[float] = []
    metric_sds: List[float] = []

    for metric_key in metric_keys:
        values = [
            float(row[metric_key])
            for row in successful
        ]
        metric_means.append(mean(values))
        metric_sds.append(
            stdev(values)
            if len(values) > 1
            else 0.0
        )

    output_root.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(8, 5))
    plt.bar(
        labels,
        metric_means,
        yerr=metric_sds,
        capsize=4,
    )
    plt.ylabel("Score")
    plt.ylim(0.0, 1.0)
    plt.title(
        f"{CROSS_YEAR_TAG}: target test mean and sample SD"
    )
    plt.tight_layout()
    plt.savefig(
        output_root / "target_test_metrics_mean_sd.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()


## Module 9: Target test checkpoint evaluator

This module evaluates a frozen final checkpoint on the target year test split only. It is called only after Random Search has selected and frozen the winning configuration.


In [ ]:
def evaluate_checkpoint_on_target_test_split(
    *,
    checkpoint_path: Path,
    dataset_yaml: Path,
    output_root: Path,
    run_name: str,
    expected_class_names: List[str],
    image_size: int,
    batch_size: int,
    workers: int,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Evaluate one frozen checkpoint on the target-year test split only."""
    checkpoint_path = Path(checkpoint_path)
    dataset_yaml = Path(dataset_yaml)
    output_root = Path(output_root)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"

    output_root.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(checkpoint_path))

    test_start = time.perf_counter()
    test_metrics = model.val(
        data=str(dataset_yaml.resolve()),
        split="test",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(output_root.resolve()),
        name=run_name,
        plots=False,
        save_json=False,
        verbose=False,
    )
    test_seconds = time.perf_counter() - test_start

    metric_data = extract_box_metrics(test_metrics, expected_class_names)

    per_class_rows = [
        {
            "class_id": row["class_id"],
            "class_name": row["class_name"],
            "target_test_map50_95": row["map50_95"],
        }
        for row in metric_data["per_class_map50_95"]
    ]

    return {
        "target_test_precision": metric_data["precision"],
        "target_test_recall": metric_data["recall"],
        "target_test_map50": metric_data["map50"],
        "target_test_map50_95": metric_data["map50_95"],
        "target_test_bottom3_mean_map50_95": metric_data["bottom3_mean_map50_95"],
        "target_test_seconds": float(test_seconds),
        "per_class_rows": per_class_rows,
    }

## Module 10: Complete cross year Random Search workflow

This module runs all 8 HPO configurations for 40 epochs each, freezes the best source validation configuration, retrains it with seeds 42, 123, and 999, evaluates the target test split, saves summaries, and removes the temporary `yolo_data` and HPO folders after all runs finish successfully.


In [ ]:
def run_cross_year_random_search_baseline() -> Dict[str, Any]:
    """Run the complete full-budget cross-year Random Search experiment."""
    workflow_start = time.perf_counter()
    selection_start = time.perf_counter()
    device = "0" if torch.cuda.is_available() else "cpu"

    # --------------------------------------------------------
    # Output files
    # --------------------------------------------------------
    plan_csv = RUN_STATS_ROOT / "random_search_plan_8x40.csv"
    hpo_results_csv = RUN_STATS_ROOT / "hpo40_source_validation_results.csv"
    ranked_hpo_csv = RUN_STATS_ROOT / "hpo40_ranked_results.csv"
    selected_json = RUN_STATS_ROOT / "selected_random_search_config.json"
    boundary_csv = RUN_STATS_ROOT / "selected_config_boundary_report.csv"

    final_csv = RUN_STATS_ROOT / "final_three_run_cross_year_results.csv"
    final_summary_csv = RUN_STATS_ROOT / "final_three_run_mean_sd_summary.csv"
    per_class_csv = RUN_STATS_ROOT / "final_three_run_target_test_per_class.csv"
    per_class_summary_csv = RUN_STATS_ROOT / "final_three_run_target_test_per_class_mean_sd.csv"
    manifest_json = RUN_STATS_ROOT / "experiment_manifest.json"

    print("=" * 96)
    print("CROSS YEAR RANDOM SEARCH: 8 CONFIGURATIONS x 40 EPOCHS")
    print("=" * 96)
    print("Direction:", CROSS_YEAR_TAG)
    print("HPO fitness:", "source-year validation mAP50-95 only")
    print("HPO budget:", HPO_EPOCH_UNITS, "epoch-units")
    print("Workers / cache:", WORKERS, "/", CACHE_MODE)
    print("Target test used during HPO: NO")
    print("Device:", device)
    print("Output root:", relative_output_path(RUN_ROOT, WORKSPACE_ROOT))
    print("=" * 96)

    # --------------------------------------------------------
    # 1. Prepare the single optimization split used by all HPO trials
    # --------------------------------------------------------
    optimization_cross, optimization_source, optimization_target, optimization_info = (
        make_cross_year_splits_for_seed(OPTIMIZATION_SPLIT_SEED)
    )

    optimization_dataset_root = YOLO_DATA_ROOT / f"opt_seed{OPTIMIZATION_SPLIT_SEED}"
    optimization_manifest_root = SPLITS_ROOT / CROSS_YEAR_TAG / f"opt_seed{OPTIMIZATION_SPLIT_SEED}"

    optimization_yaml, optimization_coverage = prepare_cross_year_yolo_dataset(
        cross_year_splits=optimization_cross,
        source_splits=optimization_source,
        target_splits=optimization_target,
        split_info=optimization_info,
        output_root=optimization_dataset_root,
        manifest_root=optimization_manifest_root,
        image_mode=IMAGE_PLACEMENT_MODE,
        seed=OPTIMIZATION_SPLIT_SEED,
    )

    # --------------------------------------------------------
    # 2. Build and audit the reproducible 8-point Random Search plan
    # --------------------------------------------------------
    random_search_plan = build_random_search_plan(
        n_trials=N_RANDOM_TRIALS,
        random_search_seed=RANDOM_SEARCH_SEED,
        search_space=SEARCH_SPACE,
        parameter_order=SEARCH_PARAMETER_ORDER,
    )

    repeated_plan = build_random_search_plan(
        n_trials=N_RANDOM_TRIALS,
        random_search_seed=RANDOM_SEARCH_SEED,
        search_space=SEARCH_SPACE,
        parameter_order=SEARCH_PARAMETER_ORDER,
    )
    if random_search_plan != repeated_plan:
        raise RuntimeError("Random Search reproducibility audit failed.")

    save_rows_to_csv_safe(random_search_plan, plan_csv)

    print("\nRANDOM SEARCH PLAN")
    for row in random_search_plan:
        print(json.dumps(row, indent=2))

    # --------------------------------------------------------
    # 3. HPO: all 8 configurations receive 40 epochs
    # --------------------------------------------------------
    hpo_results: List[Dict[str, Any]] = []

    for config in random_search_plan:
        trial_index = int(config["trial_index"])
        wall_start = time.perf_counter()

        print("\n\n" + "#" * 96)
        print(f"HPO40 TRIAL {trial_index}/{N_RANDOM_TRIALS} | {CROSS_YEAR_TAG}")
        print("#" * 96)
        print(json.dumps(config, indent=2))

        try:
            result = train_and_validate_configuration(
                learning_rate=float(config["learning_rate"]),
                lrf=float(config["lrf"]),
                momentum=float(config["momentum"]),
                weight_decay=float(config["weight_decay"]),
                scale=float(config["scale"]),
                dataset_yaml=optimization_yaml,
                project_dir=HPO_TRIALS_ROOT / f"t{trial_index:02d}",
                model_name=MODEL_NAME,
                epochs=HPO_EPOCHS,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                patience=PATIENCE,
                workers=WORKERS,
                seed=OPTIMIZATION_TRAIN_SEED,
                warmup_epochs=HPO_WARMUP_EPOCHS,
                close_mosaic=HPO_CLOSE_MOSAIC,
                fixed_training_args=FIXED_TRAINING_ARGS,
                expected_class_names=CLASS_NAMES,
                strict_fitness_alignment_check=STRICT_FITNESS_ALIGNMENT_CHECK,
                fitness_alignment_tolerance=FITNESS_ALIGNMENT_TOLERANCE,
                device=device,
            )

            row = {
                "status": "PASS",
                "cross_year_tag": CROSS_YEAR_TAG,
                "train_year": TRAIN_YEAR,
                "test_year": TEST_YEAR,
                **config,
                "optimization_split_seed": OPTIMIZATION_SPLIT_SEED,
                "optimization_train_seed": OPTIMIZATION_TRAIN_SEED,
                "hpo_epochs": HPO_EPOCHS,
                "warmup_epochs": HPO_WARMUP_EPOCHS,
                "close_mosaic": HPO_CLOSE_MOSAIC,
                "source_val_precision": float(result["metrics"]["precision"]),
                "source_val_recall": float(result["metrics"]["recall"]),
                "source_val_map50": float(result["metrics"]["map50"]),
                "source_val_map50_95": float(result["metrics"]["map50_95"]),
                "source_val_bottom3_mean_map50_95": float(
                    result["metrics"]["bottom3_mean_map50_95"]
                ),
                "fitness": float(result["metrics"]["map50_95"]),
                "fitness_alignment_status": result["fitness_alignment"]["status"],
                "trainer_best_fitness": result["fitness_alignment"]["trainer_best_fitness"],
                "max_epoch_map50_95": result["fitness_alignment"]["max_epoch_map50_95"],
                "fitness_alignment_difference": result["fitness_alignment"]["absolute_difference"],
                "training_seconds": float(result["timing"]["training_seconds"]),
                "validation_seconds": float(result["timing"]["validation_seconds"]),
                "train_val_seconds": float(result["timing"]["total_seconds"]),
                "trial_wall_seconds": float(time.perf_counter() - wall_start),
                "peak_gpu_memory_gb": result["hardware"]["peak_gpu_memory_gb"],
                "best_checkpoint": result["paths"]["best_checkpoint"],
                "results_csv": result["paths"]["results_csv"],
                "source_train_count": optimization_info["effective_cross_year_counts"]["train"],
                "source_valid_count": optimization_info["effective_cross_year_counts"]["valid"],
                "target_test_count": optimization_info["effective_cross_year_counts"]["test"],
                "subsample_enabled": optimization_info["subsample_info"]["enabled"],
                "error_message": "",
            }

        except Exception as error:
            row = {
                "status": "FAIL",
                "cross_year_tag": CROSS_YEAR_TAG,
                "train_year": TRAIN_YEAR,
                "test_year": TEST_YEAR,
                **config,
                "optimization_split_seed": OPTIMIZATION_SPLIT_SEED,
                "optimization_train_seed": OPTIMIZATION_TRAIN_SEED,
                "hpo_epochs": HPO_EPOCHS,
                "warmup_epochs": HPO_WARMUP_EPOCHS,
                "close_mosaic": HPO_CLOSE_MOSAIC,
                "source_val_precision": "",
                "source_val_recall": "",
                "source_val_map50": "",
                "source_val_map50_95": "",
                "source_val_bottom3_mean_map50_95": "",
                "fitness": "",
                "fitness_alignment_status": "FAIL",
                "trainer_best_fitness": "",
                "max_epoch_map50_95": "",
                "fitness_alignment_difference": "",
                "training_seconds": "",
                "validation_seconds": "",
                "train_val_seconds": "",
                "trial_wall_seconds": float(time.perf_counter() - wall_start),
                "peak_gpu_memory_gb": "",
                "best_checkpoint": "",
                "results_csv": "",
                "source_train_count": optimization_info["effective_cross_year_counts"]["train"],
                "source_valid_count": optimization_info["effective_cross_year_counts"]["valid"],
                "target_test_count": optimization_info["effective_cross_year_counts"]["test"],
                "subsample_enabled": optimization_info["subsample_info"]["enabled"],
                "error_message": privacy_safe_text(error, WORKSPACE_ROOT),
            }
            print("\nERROR during HPO40 trial:")
            print(row["error_message"])
            traceback.print_exc()

        hpo_results.append(row)
        save_rows_to_csv_safe(hpo_results, hpo_results_csv)

    successful_hpo = [row for row in hpo_results if row["status"] == "PASS"]

    if REQUIRE_ALL_HPO_TRIALS and len(successful_hpo) != N_RANDOM_TRIALS:
        raise RuntimeError(
            f"The full Random Search budget requires {N_RANDOM_TRIALS} successful trials, "
            f"but only {len(successful_hpo)} completed. Fix/re-run failed trials before selection."
        )
    if not successful_hpo:
        raise RuntimeError("No Random Search trials completed successfully.")

    ranked_hpo = rank_by_map50_95(
        rows=hpo_results,
        map95_key="source_val_map50_95",
        map50_key="source_val_map50",
        recall_key="source_val_recall",
    )

    for rank, row in enumerate(ranked_hpo, start=1):
        row["hpo40_rank"] = int(rank)

    save_rows_to_csv_safe(ranked_hpo, ranked_hpo_csv)
    plot_hpo_progress(hpo_results, RUN_PLOTS_ROOT)

    winner = dict(ranked_hpo[0])
    selection_seconds = time.perf_counter() - selection_start

    selected_config = {
        "method": "Cross year Random Search: 8 configurations x 40 epochs",
        "cross_year_tag": CROSS_YEAR_TAG,
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "fitness": "source validation mAP50-95",
        "target_test_used_for_selection": False,
        "selected_trial_index": int(winner["trial_index"]),
        "hpo40_rank": int(winner["hpo40_rank"]),
        "selected_source_val_map50_95": float(winner["source_val_map50_95"]),
        "selected_source_val_map50": float(winner["source_val_map50"]),
        "selected_source_val_recall": float(winner["source_val_recall"]),
        **{name: float(winner[name]) for name in SEARCH_PARAMETER_ORDER},
        "hpo_epochs_per_trial": HPO_EPOCHS,
        "hpo_epoch_units": HPO_EPOCH_UNITS,
        "random_search_seed": RANDOM_SEARCH_SEED,
        "optimization_split_seed": OPTIMIZATION_SPLIT_SEED,
        "optimization_train_seed": OPTIMIZATION_TRAIN_SEED,
        "selection_stage_seconds": float(selection_seconds),
        "optimization_split_info": optimization_info,
    }
    save_json(selected_config, selected_json)

    boundary_report = boundary_proximity_report(
        selected_config,
        SEARCH_SPACE,
        SEARCH_PARAMETER_ORDER,
        boundary_fraction=0.10,
    )
    save_rows_to_csv_safe(boundary_report, boundary_csv)

    print("\nSELECTED CONFIGURATION")
    print(json.dumps(selected_config, indent=2))
    print("\nBOUNDARY DIAGNOSTIC")
    for row in boundary_report:
        print(row)

    # --------------------------------------------------------
    # 4. Final three-seed cross-year training and target testing
    # --------------------------------------------------------
    final_start = time.perf_counter()
    final_rows: List[Dict[str, Any]] = []
    per_class_rows: List[Dict[str, Any]] = []

    for seed in FINAL_EVALUATION_SEEDS:
        final_wall_start = time.perf_counter()

        print("\n\n" + "#" * 96)
        print(f"FINAL CROSS-YEAR EVALUATION | {CROSS_YEAR_TAG} | SEED {seed}")
        print("#" * 96)

        try:
            final_cross, final_source, final_target, final_info = (
                make_cross_year_splits_for_seed(seed)
            )

            final_dataset_root = YOLO_DATA_ROOT / f"final_seed{seed}"
            final_manifest_root = SPLITS_ROOT / CROSS_YEAR_TAG / f"final_seed{seed}"

            final_yaml, final_coverage = prepare_cross_year_yolo_dataset(
                cross_year_splits=final_cross,
                source_splits=final_source,
                target_splits=final_target,
                split_info=final_info,
                output_root=final_dataset_root,
                manifest_root=final_manifest_root,
                image_mode=IMAGE_PLACEMENT_MODE,
                seed=seed,
            )

            train_result = train_and_validate_configuration(
                learning_rate=selected_config["learning_rate"],
                lrf=selected_config["lrf"],
                momentum=selected_config["momentum"],
                weight_decay=selected_config["weight_decay"],
                scale=selected_config["scale"],
                dataset_yaml=final_yaml,
                project_dir=FINAL_TRAINING_ROOT / f"seed{seed}",
                model_name=MODEL_NAME,
                epochs=FINAL_EPOCHS,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                patience=PATIENCE,
                workers=WORKERS,
                seed=seed,
                warmup_epochs=FINAL_WARMUP_EPOCHS,
                close_mosaic=FINAL_CLOSE_MOSAIC,
                fixed_training_args=FIXED_TRAINING_ARGS,
                expected_class_names=CLASS_NAMES,
                strict_fitness_alignment_check=STRICT_FITNESS_ALIGNMENT_CHECK,
                fitness_alignment_tolerance=FITNESS_ALIGNMENT_TOLERANCE,
                device=device,
            )

            target_test_result = evaluate_checkpoint_on_target_test_split(
                checkpoint_path=Path(train_result["paths"]["best_checkpoint"]),
                dataset_yaml=final_yaml,
                output_root=TEST_EVAL_ROOT / f"seed{seed}",
                run_name=f"target_test_seed{seed}",
                expected_class_names=CLASS_NAMES,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                workers=WORKERS,
                device=device,
            )

            final_row = {
                "status": "PASS",
                "cross_year_tag": CROSS_YEAR_TAG,
                "train_year": TRAIN_YEAR,
                "test_year": TEST_YEAR,
                "split_seed": int(seed),
                "selected_trial_index": selected_config["selected_trial_index"],
                **{name: selected_config[name] for name in SEARCH_PARAMETER_ORDER},
                "final_source_val_precision": train_result["metrics"]["precision"],
                "final_source_val_recall": train_result["metrics"]["recall"],
                "final_source_val_map50": train_result["metrics"]["map50"],
                "final_source_val_map50_95": train_result["metrics"]["map50_95"],
                "target_test_precision": target_test_result["target_test_precision"],
                "target_test_recall": target_test_result["target_test_recall"],
                "target_test_map50": target_test_result["target_test_map50"],
                "target_test_map50_95": target_test_result["target_test_map50_95"],
                "target_test_bottom3_mean_map50_95": (
                    target_test_result["target_test_bottom3_mean_map50_95"]
                ),
                "fitness_alignment_status": train_result["fitness_alignment"]["status"],
                "final_training_seconds": train_result["timing"]["training_seconds"],
                "final_source_validation_seconds": train_result["timing"]["validation_seconds"],
                "target_test_seconds": target_test_result["target_test_seconds"],
                "final_run_wall_seconds": float(time.perf_counter() - final_wall_start),
                "source_train_count": final_info["effective_cross_year_counts"]["train"],
                "source_valid_count": final_info["effective_cross_year_counts"]["valid"],
                "target_test_count": final_info["effective_cross_year_counts"]["test"],
                "subsample_enabled": final_info["subsample_info"]["enabled"],
                "subsample_target_size": final_info["subsample_info"]["target_size"],
                "subsample_seed": final_info["subsample_info"]["subsample_seed"],
                "best_checkpoint": train_result["paths"]["best_checkpoint"],
                "dataset_yaml": relative_output_path(final_yaml, WORKSPACE_ROOT),
                "error_message": "",
            }

            for class_row in target_test_result["per_class_rows"]:
                per_class_rows.append({
                    "cross_year_tag": CROSS_YEAR_TAG,
                    "train_year": TRAIN_YEAR,
                    "test_year": TEST_YEAR,
                    "split_seed": int(seed),
                    **class_row,
                })

        except Exception as error:
            final_row = {
                "status": "FAIL",
                "cross_year_tag": CROSS_YEAR_TAG,
                "train_year": TRAIN_YEAR,
                "test_year": TEST_YEAR,
                "split_seed": int(seed),
                "selected_trial_index": selected_config["selected_trial_index"],
                **{name: selected_config[name] for name in SEARCH_PARAMETER_ORDER},
                "final_source_val_precision": "",
                "final_source_val_recall": "",
                "final_source_val_map50": "",
                "final_source_val_map50_95": "",
                "target_test_precision": "",
                "target_test_recall": "",
                "target_test_map50": "",
                "target_test_map50_95": "",
                "target_test_bottom3_mean_map50_95": "",
                "fitness_alignment_status": "FAIL",
                "final_training_seconds": "",
                "final_source_validation_seconds": "",
                "target_test_seconds": "",
                "final_run_wall_seconds": float(time.perf_counter() - final_wall_start),
                "source_train_count": "",
                "source_valid_count": "",
                "target_test_count": "",
                "subsample_enabled": "",
                "subsample_target_size": "",
                "subsample_seed": "",
                "best_checkpoint": "",
                "dataset_yaml": "",
                "error_message": privacy_safe_text(error, WORKSPACE_ROOT),
            }
            print("\nERROR during final cross year evaluation:")
            print(final_row["error_message"])
            traceback.print_exc()

        final_rows.append(final_row)
        save_rows_to_csv_safe(final_rows, final_csv)
        save_rows_to_csv_safe(per_class_rows, per_class_csv)

    final_seconds = time.perf_counter() - final_start
    total_seconds = time.perf_counter() - workflow_start

    successful_final = [row for row in final_rows if row["status"] == "PASS"]
    if len(successful_final) != len(FINAL_EVALUATION_SEEDS):
        raise RuntimeError(
            f"All {len(FINAL_EVALUATION_SEEDS)} final runs are required; "
            f"only {len(successful_final)} completed successfully."
        )

    # --------------------------------------------------------
    # 5. Summaries, final plot, and experiment manifest
    # --------------------------------------------------------
    summary_metrics = [
        "final_source_val_precision",
        "final_source_val_recall",
        "final_source_val_map50",
        "final_source_val_map50_95",
        "target_test_precision",
        "target_test_recall",
        "target_test_map50",
        "target_test_map50_95",
        "target_test_bottom3_mean_map50_95",
        "final_training_seconds",
        "final_source_validation_seconds",
        "target_test_seconds",
        "final_run_wall_seconds",
    ]
    summary_rows = summarize_metric_mean_sd(successful_final, summary_metrics)

    summary_rows.extend([
        {
            "metric": "hpo_epoch_units",
            "n": 1,
            "mean": HPO_EPOCH_UNITS,
            "sd": "",
            "min": HPO_EPOCH_UNITS,
            "max": HPO_EPOCH_UNITS,
        },
        {
            "metric": "final_evaluation_epoch_units",
            "n": 1,
            "mean": FINAL_EVALUATION_EPOCH_UNITS,
            "sd": "",
            "min": FINAL_EVALUATION_EPOCH_UNITS,
            "max": FINAL_EVALUATION_EPOCH_UNITS,
        },
        {
            "metric": "selection_stage_wall_seconds",
            "n": 1,
            "mean": selection_seconds,
            "sd": "",
            "min": selection_seconds,
            "max": selection_seconds,
        },
        {
            "metric": "final_stage_wall_seconds",
            "n": 1,
            "mean": final_seconds,
            "sd": "",
            "min": final_seconds,
            "max": final_seconds,
        },
        {
            "metric": "total_workflow_wall_seconds",
            "n": 1,
            "mean": total_seconds,
            "sd": "",
            "min": total_seconds,
            "max": total_seconds,
        },
    ])

    per_class_summary = summarize_per_class_mean_sd(per_class_rows)

    save_rows_to_csv_safe(summary_rows, final_summary_csv)
    save_rows_to_csv_safe(per_class_summary, per_class_summary_csv)
    plot_final_target_test_metrics(final_rows, RUN_PLOTS_ROOT)

    manifest = {
        "method": "Cross year Random Search: 8 configurations x 40 epochs",
        "cross_year_tag": CROSS_YEAR_TAG,
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "model": MODEL_NAME,
        "fitness": "source validation mAP50-95",
        "target_test_used_for_selection": False,
        "search_space_version": SEARCH_SPACE_VERSION,
        "search_space": SEARCH_SPACE,
        "search_parameter_order": SEARCH_PARAMETER_ORDER,
        "reference_untuned_values": REFERENCE_UNTUNED_VALUES,
        "fixed_training_args": FIXED_TRAINING_ARGS,
        "runtime": {
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "workers": WORKERS,
            "cache": CACHE_MODE,
            "patience": PATIENCE,
            "image_placement_mode": IMAGE_PLACEMENT_MODE,
        },
        "subsampling_protocol": {
            "enabled_only_when_source_year_2021": SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE,
            "target_mode": SUBSAMPLE_TARGET_MODE,
            "seed_offset": SUBSAMPLE_SEED_OFFSET,
            "method": "image level primary label stratified subsampling",
        },
        "hpo_protocol": {
            "trials": N_RANDOM_TRIALS,
            "epochs_per_trial": HPO_EPOCHS,
            "warmup_epochs": HPO_WARMUP_EPOCHS,
            "close_mosaic": HPO_CLOSE_MOSAIC,
            "epoch_units": HPO_EPOCH_UNITS,
            "random_search_seed": RANDOM_SEARCH_SEED,
            "optimization_split_seed": OPTIMIZATION_SPLIT_SEED,
            "optimization_train_seed": OPTIMIZATION_TRAIN_SEED,
        },
        "final_evaluation": {
            "seeds": FINAL_EVALUATION_SEEDS,
            "epochs": FINAL_EPOCHS,
            "warmup_epochs": FINAL_WARMUP_EPOCHS,
            "close_mosaic": FINAL_CLOSE_MOSAIC,
            "epoch_units": FINAL_EVALUATION_EPOCH_UNITS,
        },
        "fitness_alignment_check": {
            "strict": STRICT_FITNESS_ALIGNMENT_CHECK,
            "tolerance": FITNESS_ALIGNMENT_TOLERANCE,
        },
        "selected_config": selected_config,
        "optimization_split_info": optimization_info,
        "optimization_class_coverage": optimization_coverage,
        "software": {
            "python": sys.version.split()[0],
            "ultralytics": ultralytics.__version__,
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_version": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        },
        "timing": {
            "selection_stage_seconds": float(selection_seconds),
            "final_stage_seconds": float(final_seconds),
            "total_workflow_seconds": float(total_seconds),
        },
        "output_root": relative_output_path(RUN_ROOT, WORKSPACE_ROOT),
    }
    save_json(manifest, manifest_json)

    print("\n" + "=" * 96)
    print("CROSS YEAR RANDOM SEARCH COMPLETE")
    print("=" * 96)
    print("Direction:", CROSS_YEAR_TAG)
    print("Selected trial:", selected_config["selected_trial_index"])
    print(
        "Selected source validation mAP50-95:",
        f"{selected_config['selected_source_val_map50_95']:.6f}",
    )
    print("HPO budget:", HPO_EPOCH_UNITS, "epoch-units")
    print("Final evaluation budget:", FINAL_EVALUATION_EPOCH_UNITS, "epoch-units")
    print("Successful final runs:", len(successful_final), "/", len(FINAL_EVALUATION_SEEDS))
    print("Target year test data was used only after the configuration was frozen.")

    print("\nFinal mean and sample SD:")
    for row in summary_rows:
        if row["mean"] == "":
            continue
        if row["sd"] == "":
            print(f"{row['metric']}: {float(row['mean']):.4f}")
        else:
            print(f"{row['metric']}: {float(row['mean']):.4f} +/- {float(row['sd']):.4f}")

    print("\nSaved files:")
    for path in [
        plan_csv,
        hpo_results_csv,
        ranked_hpo_csv,
        selected_json,
        boundary_csv,
        final_csv,
        final_summary_csv,
        per_class_csv,
        per_class_summary_csv,
        manifest_json,
    ]:
        print(relative_output_path(path, WORKSPACE_ROOT))
    print("=" * 96)

    # Remove large temporary folders only after every HPO and final run succeeds.
    if YOLO_DATA_ROOT.exists():
        shutil.rmtree(YOLO_DATA_ROOT)

    if HPO_TRIALS_ROOT.exists():
        shutil.rmtree(HPO_TRIALS_ROOT)

    print("\nCleanup completed:")
    print(
        "Removed:",
        relative_output_path(YOLO_DATA_ROOT, WORKSPACE_ROOT),
    )
    print(
        "Removed:",
        relative_output_path(HPO_TRIALS_ROOT, WORKSPACE_ROOT),
    )

    return {
        "selected_config": selected_config,
        "hpo_results": hpo_results,
        "ranked_hpo": ranked_hpo,
        "final_rows": final_rows,
        "summary_rows": summary_rows,
        "per_class_summary_rows": per_class_summary,
        "selection_stage_seconds": float(selection_seconds),
        "final_stage_seconds": float(final_seconds),
        "total_workflow_seconds": float(total_seconds),
        "output_root": relative_output_path(RUN_ROOT, WORKSPACE_ROOT),
    }


## Module 11: Run switch

Keep the switch `False` while reviewing the notebook. Set it to `True` only when the datasets are in place and you are ready to run the complete 320 epoch HPO stage followed by the three final 40 epoch training runs.


In [ ]:
RUN_CROSS_YEAR_RANDOM_SEARCH = False

if RUN_CROSS_YEAR_RANDOM_SEARCH:
    cross_year_random_search_result = run_cross_year_random_search_baseline()
else:
    print("RUN_CROSS_YEAR_RANDOM_SEARCH is False.")
    print(f"Direction: {CROSS_YEAR_TAG}")
    print(f"HPO:   {N_RANDOM_TRIALS} x {HPO_EPOCHS} = {HPO_EPOCH_UNITS} epoch-units")
    print(
        f"Final: {len(FINAL_EVALUATION_SEEDS)} x {FINAL_EPOCHS} = "
        f"{FINAL_EVALUATION_EPOCH_UNITS} epoch-units (reported separately)"
    )
    print("Fitness: source validation mAP50-95 only")
    print("Target year test is not used until the HPO winner is frozen.")
    print("Runtime: workers=8, cache='disk', batch=16, imgsz=640, patience=0")
    print("Set RUN_CROSS_YEAR_RANDOM_SEARCH = True when ready.")


### Comparison protocol

For fair comparison with Hyperband, DEHB, and the proposed method, keep the data protocol, 5D search domain, fitness criterion, fixed YOLO settings, optimization split, Year2021 source subsampling rule, and final seeds unchanged. Report the 320 HPO epoch units separately from the 120 final evaluation epoch units.
